In [2]:
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
from pathlib import Path

def string_to_xml_file(xml_string, file_name):
    """
    Converts a string into a well-formatted (indented) XML file, without unnecessary newlines.

    Parameters:
    xml_string (str): The XML content as a string.
    file_name (str): The desired filename for the XML file.
    """
    try:
        # Parse the XML string
        root = ET.ElementTree(ET.fromstring(xml_string))
        
        # Convert ElementTree to a string
        rough_string = ET.tostring(root.getroot(), encoding="utf-8")
        
        # Use minidom to pretty-print the XML
        parsed = minidom.parseString(rough_string)
        pretty_xml_as_string = parsed.toprettyxml(indent="  ")
        
        # Remove unnecessary blank lines created by toprettyxml()
        pretty_xml_as_string = "\n".join([line for line in pretty_xml_as_string.splitlines() if line.strip()])
        
        # Write the formatted XML to a file
        with open(file_name, "w", encoding="utf-8") as f:
            f.write(pretty_xml_as_string)
        
        print(f"XML file '{file_name}' created successfully with proper indentation and no extra newlines.")
    except ET.ParseError as e:
        print("Error parsing XML string:", e)

# Source

## Data Sources
- Ministry of Agriculture, Food and Rural Affairs (MAFRA), 2021, 2050 Carbon Neutrality Strategy for the Agriculture and Food Sector. (`../resources/CNSAF-2021-MAFRA`)

## Implemented Input Files
- `/input/gcamdata/xml/N_Fert_reduction.xml`

# Nitrogen Fertilizer Reduction Policy for Cropland

South Korea plans to reduce nitrogen fertilizer application rates from **262 kg/ha** (2019 baseline) to **115 kg/ha**, representing a **43.9% reduction**.  

GWP reduction rates are applied to the default GCAM emissions coefficients to ensure model consistency.  
Below are calculations for implementing the policy.

In [30]:
def modify_fertilizer_coefficient(input_file, output_file):
    """
    Reduces the N fertilizer coefficient by 56.1% for all agricultural
    production technologies in South Korea from the year 2030 onwards.

    Args:
        input_file (str): The path to the input XML file.
        output_file (str): The path to the output XML file.
    """
    tree = ET.parse(input_file)
    root = tree.getroot()

    # Find the region "South Korea"
    for region in root.findall('.//region[@name="South Korea"]'):
        # Iterate through all AgProductionTechnology elements
        for tech in region.findall('.//AgProductionTechnology'):
            # Find all period elements from 2030 onwards
            for period in tech.findall('period'):
                year = int(period.get('year'))
                if year >= 2030:
                    # Find the N fertilizer input
                    for n_fertilizer in period.findall('.//minicam-energy-input[@name="N fertilizer"]'):
                        # Get the coefficient and reduce it
                        coefficient_element = n_fertilizer.find('coefficient')
                        if coefficient_element is not None:
                            original_coefficient = float(coefficient_element.text)
                            new_coefficient = original_coefficient * 0.561
                            coefficient_element.text = str(new_coefficient)

    xml_string = ET.tostring(root, encoding="unicode")
    string_to_xml_file(xml_string, output_file)

In [ ]:
input_file = "../../input/gcamdata/xml/ag_Fert_IRR_MGMT.xml"
output_file = "../../input/policy/korea-2035/agriculture/N_Fert_reduction.xml"

In [32]:
modify_fertilizer_coefficient(input_file, output_file)

XML file '/home/hyuntae-choi/gcam-core/input/policy/korea-2035/agriculture/N_Fert_reduction.xml' created successfully with proper indentation and no extra newlines.
